# RSRP Dataset — Data Preparation  ·  v3

**Project:** Crowd Network Intelligence — 4G Operator Benchmarking
**Dataset:** `RSRP.csv` — 2,725,353 rows × 11 columns
**Coverage:** Riyadh, Saudi Arabia · 2–4 November 2019

---

## What changed from v2

The v2 cleaning rules were sound and are carried over **unchanged**. Three things are new:

| # | Addition | Why |
|---|---|---|
| **1** | **Canonical device names** (§B.4) | v2 removed the duplicate *rows* but kept whichever alias came first, so `Galaxy J7 Prime` and `Galaxy On Nxt` both survived as separate devices — the same phone drawn twice, its samples split between them. |
| **2** | **Quantised-device detection** (§B.5) | A handful of models report the same two or three RSRP values over and over instead of measuring. One of them, `Galaxy Note8` on Operator A, is 19% of that operator's entire sample base and drags its average down by ~2 dB. v2 never looked for this. |
| **3** | **H3 hex keys alongside the degree bins** (§C.2) | Degree bins stay the primary grid. H3 is carried as a second key so a future dashboard can switch grid without reprocessing. |

**Nothing is deleted on a judgement call.** Rows that *cannot* be a valid 4G measurement are
removed; rows that are merely *suspect* are flagged, so every downstream number can be shown
both ways.

**Pipeline order:** scope reduction → normalisation → derivation. Cheap filters run before
expensive scans; derived fields are computed last, after the row set is final.

---
## Section 0 — Environment Setup

Imports, paths, and helper utilities. No data operations; re-runnable at any point.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Standard library
# ─────────────────────────────────────────────────────────────────────────────
import gc
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# Third-party
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

try:
    import h3                      # secondary spatial key only; degree bins are primary
    HAS_H3 = True
except ImportError:
    HAS_H3 = False
    print("NOTE: h3 not installed — the H3 columns will be skipped.")
    print("      Install with:  pip install h3     (degree bins work regardless)")

# `display` exists in Jupyter; this fallback keeps the notebook runnable as a plain script
try:
    display
except NameError:
    def display(x): print(x.to_string() if hasattr(x, "to_string") else x)

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"h3     : {'yes' if HAS_H3 else 'not installed'}")

pandas : 3.0.2
numpy  : 2.4.4
h3     : yes


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Paths — resolved relative to this notebook, so the project folder can move
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR   = Path.cwd().parent if Path.cwd().name.lower() == "code" else Path.cwd()
DATA_DIR   = BASE_DIR / "Data"

# ─────────────────────────────────────────────────────────────────────────────
# Raw inputs. Look in this project's Data folder first; fall back to the v3
# folder next door, so v4 works whether or not you copied the 334 MB RSRP.csv
# across. Nothing is written to v3 — it is read-only here.
# ─────────────────────────────────────────────────────────────────────────────
def find_raw(filename):
    for cand in (DATA_DIR / filename,
                 BASE_DIR.parent / "v3" / "Data" / filename):
        if cand.exists():
            return cand
    raise FileNotFoundError(
        f"{filename} not found in {DATA_DIR} or {BASE_DIR.parent / 'v3' / 'Data'}. "
        f"Copy the raw file into Data/ and re-run."
    )

RSRP_PATH = None   # resolved after the output folders are created
# ─────────────────────────────────────────────────────────────────────────────
# Output folders — one job each, so the folder Tableau opens contains only files
# Tableau should load
#
#   interim/    regenerable working files (parquet). Notebook 03 reads them;
#               nothing else ever should. Safe to delete and re-run.
#   processed/  the nine flat files Tableau loads. Nothing else goes here.
#   reference/  the audit trail — cleaning logs, the alias map. Documented
#               evidence, never a data source.
# ─────────────────────────────────────────────────────────────────────────────
INTERIM_DIR = DATA_DIR / "interim"
OUT_DIR     = DATA_DIR / "processed"
REF_DIR     = DATA_DIR / "reference"
for _d in (INTERIM_DIR, OUT_DIR, REF_DIR):
    _d.mkdir(parents=True, exist_ok=True)

RSRP_PATH = find_raw("RSRP.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ─────────────────────────────────────────────────────────────────────────────
# Domain constants
# ─────────────────────────────────────────────────────────────────────────────
RSRP_VALID_MIN = -140            # dBm, worst usable LTE reading (3GPP TS 36.133)
RSRP_VALID_MAX =  -40            # dBm, best usable LTE reading
RSRP_SENTINEL  = 2147483647      # int32 max — SDK "no reading available" marker
LOCAL_TZ       = "Asia/Riyadh"   # UTC+03:00

# The three complete days. Identical to Notebook 02 so both tables cover the same
# span — a shared date filter must not empty one dashboard while others show data.
DATE_MIN, DATE_MAX = "2019-11-02", "2019-11-04"

# Riyadh bounding box, from the observed extent of the raw data
LAT_MIN, LAT_MAX = 24.40, 25.10
LON_MIN, LON_MAX = 46.30, 47.25

# In-scope technology: RSRP is defined only for LTE
SCOPE_GENERATION = "4G"
SCOPE_CONNECTION = "Mobile"
SCOPE_DATA_STATE = "Enabled"

# Fields identifying a co-located measurement
COLOCATION_KEY = ["Timestamp", "LocationLatitude", "LocationLongitude", "RadioOperatorName"]

# Manufacturer alias map
MANUFACTURER_MAP = {
    "samsung": "Samsung", "SAMSUNG": "Samsung",
    "htc": "HTC", "HTC": "HTC",
    "lenovo": "Lenovo", "LENOVO": "Lenovo", "Lenovo": "Lenovo",
    "realme": "Realme", "Realme": "Realme",
    "LGE": "LG",
    "HMD Global": "Nokia",          # HMD manufactures Nokia-branded handsets
    "HUAWEI": "Huawei",
    "motorola": "Motorola",
    "vivo": "Vivo",
    "OPPO": "Oppo",
    "PANASONIC": "Panasonic",
    "SHARP": "Sharp",
}

# ── Quantisation detector (new in v3, §B.5) ──────────────────────────────────
# A model/operator pair is flagged when it puts at least QUANT_TOP3_SHARE of its
# readings on just its three most common RSRP values. The healthy fleet sits near
# 11%; the flagged pairs sit at 41-87%. The line is drawn in that gap.
QUANT_TOP3_SHARE  = 0.40
QUANT_MIN_SAMPLES = 3000         # do not accuse a model we barely observed

# ── Coverage bands (3GPP-conventional service tiers) ─────────────────────────
# Numeric prefix so any tool sorts them best-to-worst without extra configuration.
BAND_EDGES  = [-141, -110, -100, -90, -80, -39]
BAND_LABELS = ["5 Very Poor", "4 Poor", "3 Fair", "2 Good", "1 Excellent"]

print(f"Base dir    : {BASE_DIR}")
print(f"Source      : {RSRP_PATH}")
print(f"Interim dir : {INTERIM_DIR}")
print(f"Output dir  : {OUT_DIR}")
print(f"Reference   : {REF_DIR}")

Base dir    : /root/v5
Source      : /root/v3/Data/RSRP.csv
Interim dir : /root/v5/Data/interim
Output dir  : /root/v5/Data/processed
Reference   : /root/v5/Data/reference


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Cleaning audit trail
#
# Every step appends one entry. Exported at the end as evidence that each removal
# followed a documented rule rather than an ad-hoc call.
# ─────────────────────────────────────────────────────────────────────────────
cleaning_log = []


def log_step(step_id, rule, rows_before, rows_after, rationale, kind="DROP"):
    # Record a cleaning step and print a readable summary.
    removed = rows_before - rows_after
    pct     = (100 * removed / rows_before) if rows_before else 0.0

    cleaning_log.append({
        "step": step_id, "kind": kind, "rule": rule,
        "rows_before": rows_before, "rows_after": rows_after,
        "removed": removed, "pct_removed": round(pct, 3),
        "rationale": rationale,
    })

    print(f"[{step_id}] ({kind}) {rule}")
    print(f"    before : {rows_before:,}")
    print(f"    after  : {rows_after:,}")
    print(f"    removed: {removed:,}  ({pct:.3f}%)")


def show_log():
    return pd.DataFrame(cleaning_log)[
        ["step", "kind", "rule", "rows_before", "rows_after", "removed", "pct_removed"]
    ]

---
## Stage A — Scope Reduction

Cheapest, highest-yield filters first. Each step shrinks the frame before the next runs.

**Order rationale:** technology scoping precedes deduplication because it removes ~17% of
rows outright — including 98% of all invalid RSRP readings — making the subsequent full-row
duplicate scan cheaper and its count more meaningful.

### A.1 Load the raw dataset

All columns loaded as `str` to defer type inference:

- The RSRP sentinel and genuine readings share one column; explicit conversion later makes
  the distinction visible rather than implicit.
- Timestamps carry a UTC offset that varies by record. String loading preserves it for
  inspection before parsing.

In [4]:
rsrp = pd.read_csv(
    RSRP_PATH,
    dtype=str,               # defer type inference to explicit later steps
    keep_default_na=False,   # do not treat "NA"/"null" strings as missing
    na_values=[""],          # only genuinely empty fields count as null
)

ROWS_ORIGINAL = len(rsrp)

print(f"Rows    : {ROWS_ORIGINAL:,}")
print(f"Columns : {len(rsrp.columns)}")
print(f"\n{list(rsrp.columns)}")
rsrp.head(3)

Rows    : 2,725,353
Columns : 11

['Timestamp', 'LocationLatitude', 'LocationLongitude', 'RadioConnectionType', 'Country', 'RadioNetworkGeneration', 'RadioOperatorName', 'RSRP', 'RadioMobileDataEnabled', 'DeviceManufacturer', 'DeviceName']


,Timestamp,LocationLatitude,LocationLongitude,RadioConnectionType,Country,RadioNetworkGeneration,RadioOperatorName,RSRP,RadioMobileDataEnabled,DeviceManufacturer,DeviceName
0,2019-11-02 00:09:29.386 +0300,24.54054,46.784737,Mobile,Saudi Arabia,4G,Operator C,2147483647,Enabled,HTC,D728W
1,2019-11-02 00:09:29.386 +0300,24.54054,46.784737,Mobile,Saudi Arabia,4G,Operator C,2147483647,Enabled,HTC,Desire 728 Dual Sim
2,2019-11-02 00:10:45.323 +0300,24.54054,46.784737,Mobile,Saudi Arabia,3G,Operator C,2147483647,Enabled,HTC,D728W


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Technology mix BEFORE scoping.
#
# Captured here because step A.3 is about to delete the evidence. The dashboard's
# "LTE attach rate" KPI — what share of a user's sessions actually ran on 4G — can
# only be computed from these pre-scoping counts. Exported in Notebook 03.
# ─────────────────────────────────────────────────────────────────────────────
tech_mix = (
    rsrp.groupby(["RadioOperatorName", "RadioNetworkGeneration"])
        .size().rename("n_rows").reset_index()
)
tech_mix["pct_of_operator"] = (
    100 * tech_mix["n_rows"] / tech_mix.groupby("RadioOperatorName")["n_rows"].transform("sum")
).round(2)

print("Technology mix per operator (pre-scoping) — needed for the LTE attach-rate KPI\n")
print(tech_mix.pivot(index="RadioOperatorName",
                     columns="RadioNetworkGeneration",
                     values="pct_of_operator").fillna(0).to_string())

Technology mix per operator (pre-scoping) — needed for the LTE attach-rate KPI

RadioNetworkGeneration    2G     3G     4G  Unknown  WiFi
RadioOperatorName                                        
Operator A              0.26  15.88  83.18     0.68  0.00
Operator B              0.07  22.59  76.65     0.68  0.01
Operator C              0.21  10.38  88.91     0.50  0.00


In [6]:
print("Network generation");     print(rsrp["RadioNetworkGeneration"].value_counts().to_string(), "\n")
print("Connection type");        print(rsrp["RadioConnectionType"].value_counts().to_string(), "\n")
print("Mobile data state");      print(rsrp["RadioMobileDataEnabled"].value_counts().to_string(), "\n")
print("Operator");               print(rsrp["RadioOperatorName"].value_counts().to_string(), "\n")

nulls = rsrp.isna().sum()
print("Nulls per column")
print(nulls[nulls > 0].to_string() if nulls.any() else "  none")

Network generation
RadioNetworkGeneration
4G         2257240
3G          445246
Unknown      17361
2G            5464
WiFi            42 

Connection type
RadioConnectionType
Mobile     2717295
WiFi          7390
Unknown        668 

Mobile data state
RadioMobileDataEnabled
Enabled     2724635
Disabled        718 

Operator
RadioOperatorName
Operator A    1423592
Operator B     688448
Operator C     613313 

Nulls per column
Country    156


### A.3 Technology scoping

RSRP is defined only for LTE. It is the average power of the cell-specific reference signals,
which exist solely in the 4G downlink frame structure. Earlier technologies use entirely
different coverage measurements — 3G reports RSCP, 2G reports RxLev.

A record tagged 3G therefore **cannot** carry a valid RSRP by definition. Rows on a WiFi
connection are likewise excluded: they describe no cellular link at all.

This is a technical scoping decision, not a statistical one.

In [7]:
rows_before = len(rsrp)

scope_mask = (
    (rsrp["RadioNetworkGeneration"] == SCOPE_GENERATION) &
    (rsrp["RadioConnectionType"]    == SCOPE_CONNECTION) &
    (rsrp["RadioMobileDataEnabled"] == SCOPE_DATA_STATE)
)

rsrp = rsrp[scope_mask].reset_index(drop=True)

log_step(
    "3.2",
    f"Retain {SCOPE_GENERATION} + {SCOPE_CONNECTION} + data {SCOPE_DATA_STATE} only",
    rows_before, len(rsrp),
    "RSRP is an LTE-specific KPI (3GPP TS 36.214). Non-4G records cannot carry a "
    "valid reading; WiFi records describe no cellular link.",
)

[3.2] (DROP) Retain 4G + Mobile + data Enabled only
    before : 2,725,353
    after  : 2,250,966
    removed: 474,387  (17.406%)


### A.4 Exact duplicate removal

Rows identical across **all 11 columns** — collection-SDK retransmission artifacts carrying
no information.

Kept deliberately separate from the aliasing problem in §B.4. They are different defects:

| Type | Definition | Handled in |
|---|---|---|
| **Exact** | All 11 columns identical | Here — remove |
| **Alias** | Same time + place + operator + RSRP, *different* device name | §B.4 — rename, then collapse |

In [8]:
n_exact     = rsrp.duplicated(keep="first").sum()
n_colocated = rsrp.duplicated(subset=COLOCATION_KEY, keep="first").sum()

print("Duplication baseline (post-scoping)")
print("-" * 66)
print(f"Exact full-row duplicates : {n_exact:>9,}   -> remove here")
print(f"Co-located duplicates     : {n_colocated:>9,}   -> investigated in B.4")

Duplication baseline (post-scoping)
------------------------------------------------------------------
Exact full-row duplicates :     1,677   -> remove here
Co-located duplicates     :   101,030   -> investigated in B.4


In [9]:
rows_before = len(rsrp)
rsrp = rsrp.drop_duplicates(keep="first").reset_index(drop=True)

log_step(
    "6.1", "Drop exact full-row duplicates (all 11 columns identical)",
    rows_before, len(rsrp),
    "Byte-identical records are retransmission artifacts. Unambiguous — requires no "
    "interpretation of device, location, or timing semantics.",
)

assert rsrp.duplicated().sum() == 0, "Exact duplicates remain"
print("\nVerified: zero exact duplicates remain.")

[6.1] (DROP) Drop exact full-row duplicates (all 11 columns identical)
    before : 2,250,966
    after  : 2,249,289
    removed: 1,677  (0.075%)



Verified: zero exact duplicates remain.


### A.5 RSRP validity

Two invalid classes, removed together but counted separately:

| Class | Value | Meaning |
|---|---|---|
| **Sentinel** | `2147483647` | int32 max — SDK marker for "no reading available" |
| **Uninitialised** | `0`, `-1`, `12` | Memory artifacts, not weak signal |

Valid LTE range is **−140 to −40 dBm** per 3GPP TS 36.133.

Because scoping has already removed non-4G records, the sentinel rate observed here (~8%)
reflects genuine handset read failures rather than the 23.6% pre-scoping figure, which was
dominated by 3G rows where the measurement does not exist.

In [10]:
rsrp["RSRP"] = pd.to_numeric(rsrp["RSRP"], errors="coerce")

n_sentinel = (rsrp["RSRP"] == RSRP_SENTINEL).sum()
n_nan      = rsrp["RSRP"].isna().sum()
oor_mask   = (~rsrp["RSRP"].between(RSRP_VALID_MIN, RSRP_VALID_MAX)
              & (rsrp["RSRP"] != RSRP_SENTINEL) & rsrp["RSRP"].notna())

print(f"Sentinel ({RSRP_SENTINEL})   : {n_sentinel:>8,}  ({100*n_sentinel/len(rsrp):.2f}%)")
print(f"Unparseable / null           : {n_nan:>8,}")
print(f"Out-of-range (non-sentinel)  : {oor_mask.sum():>8,}")
print(f"\nOut-of-range values observed : {sorted(rsrp.loc[oor_mask, 'RSRP'].unique())}")

Sentinel (2147483647)   :  186,648  (8.30%)
Unparseable / null           :        0
Out-of-range (non-sentinel)  :      431

Out-of-range values observed : [np.int64(-1), np.int64(0), np.int64(12)]


In [11]:
rows_before = len(rsrp)
rsrp = rsrp[rsrp["RSRP"].between(RSRP_VALID_MIN, RSRP_VALID_MAX)].reset_index(drop=True)

log_step(
    "4.1-4.3", f"Retain RSRP within [{RSRP_VALID_MIN}, {RSRP_VALID_MAX}] dBm",
    rows_before, len(rsrp),
    "Removes the int32 sentinel (no reading available) and uninitialised values "
    "(0, -1, 12). Valid range per 3GPP TS 36.133.",
)

print("\nSurviving RSRP distribution:")
print(rsrp["RSRP"].describe().to_string())

[4.1-4.3] (DROP) Retain RSRP within [-140, -40] dBm
    before : 2,249,289
    after  : 2,062,210
    removed: 187,079  (8.317%)

Surviving RSRP distribution:
count    2.062210e+06
mean    -8.411511e+01
std      1.239343e+01
min     -1.400000e+02
25%     -9.300000e+01
50%     -8.300000e+01
75%     -7.500000e+01
max     -4.400000e+01


### A.6 Coordinate validity

Confirms every point falls inside the Riyadh study area. Expected to remove nothing — run as
an assertion against silent corruption rather than as a corrective filter.

In [12]:
rsrp["LocationLatitude"]  = pd.to_numeric(rsrp["LocationLatitude"],  errors="coerce")
rsrp["LocationLongitude"] = pd.to_numeric(rsrp["LocationLongitude"], errors="coerce")

rows_before = len(rsrp)
geo_mask = (rsrp["LocationLatitude"].between(LAT_MIN, LAT_MAX) &
            rsrp["LocationLongitude"].between(LON_MIN, LON_MAX))
rsrp = rsrp[geo_mask].reset_index(drop=True)

log_step(
    "2.1-2.2", "Retain coordinates within the Riyadh bounding box",
    rows_before, len(rsrp),
    "Guards against null, zero, and Null Island coordinates. Expected to remove zero "
    "rows — functions as a corruption assertion.",
)

[2.1-2.2] (DROP) Retain coordinates within the Riyadh bounding box
    before : 2,062,210
    after  : 2,062,210
    removed: 0  (0.000%)


---
## Stage B — Normalisation

Values are corrected in place. Rows are removed only where an investigation concludes they
must be.

### B.1 Timestamp parsing and timezone normalisation

The dataset is assumed to be Arabia Standard Time. It is not entirely — several thousand
records carry offsets ranging from `+0800` to `-0700`, despite every coordinate falling
inside Riyadh. These are **misconfigured device clocks**, not travelling users.

Parsing with `utc=True` respects each record's declared offset before converting to a common
timezone. Truncating the offset string instead would place those records in the wrong hour —
silently corrupting the hourly density view, with no later check able to detect it.

In [13]:
offsets = rsrp["Timestamp"].str[-5:].value_counts()
print("UTC offsets present in the data:")
print(offsets.to_string())
print(f"\nRecords with a non-standard offset: {len(rsrp) - offsets.get('+0300', 0):,}")

UTC offsets present in the data:
Timestamp
+0300    2057813
+0800       2955
+0400        894
+0600        185
+0200        136
+0500        125
+0530        101
-0700          1

Records with a non-standard offset: 4,397


In [14]:
rsrp["Timestamp"] = pd.to_datetime(rsrp["Timestamp"], utc=True, errors="coerce")
print(f"Unparseable timestamps: {rsrp['Timestamp'].isna().sum():,}")

rsrp["Timestamp"] = rsrp["Timestamp"].dt.tz_convert(LOCAL_TZ)

print(f"\nRange: {rsrp['Timestamp'].min()}  ->  {rsrp['Timestamp'].max()}")
print("\nRecords per day:")
print(rsrp["Timestamp"].dt.date.value_counts().sort_index().to_string())

rows_before = len(rsrp)
rsrp = rsrp[rsrp["Timestamp"].notna()].reset_index(drop=True)
log_step(
    "1.1-1.2", "Parse timestamps honouring declared UTC offset; drop unparseable",
    rows_before, len(rsrp),
    "Offsets other than +0300 originate from misconfigured device clocks. Parsing with "
    "utc=True then converting to Asia/Riyadh assigns the correct local hour.",
)

Unparseable timestamps: 0

Range: 2019-11-01 21:15:32.097000+03:00  ->  2019-11-04 23:59:59.414000+03:00

Records per day:


Timestamp
2019-11-01        64
2019-11-02    698504
2019-11-03    748293
2019-11-04    615349
[1.1-1.2] (DROP) Parse timestamps honouring declared UTC offset; drop unparseable
    before : 2,062,210
    after  : 2,062,210
    removed: 0  (0.000%)


### B.2 Boundary days

After timezone conversion, records appear on **four** dates rather than three: 64 land on
1 November, carried across the date boundary by the shift from UTC to Asia/Riyadh.

They are not a day. Retaining 64 records alongside days of ~700,000 would put a fourth bar on
every date-grouped chart, sized at roughly 0.01% of its neighbours, and add a fourth value to
every date filter. Notebook 02 applies the same bounds, so both tables end on the same days.

In [15]:
rows_before = len(rsrp)
date_mask = rsrp["Timestamp"].dt.date.astype(str).between(DATE_MIN, DATE_MAX)
rsrp = rsrp[date_mask].reset_index(drop=True)

log_step(
    "1.4", f"Restrict to complete days {DATE_MIN} to {DATE_MAX}",
    rows_before, len(rsrp),
    "Boundary-day records are timezone-conversion artifacts, not a day of measurement. "
    "Matches Notebook 02 step 1.4 so both tables share one date domain.",
)

print("\nRecords per date after restriction:")
print(rsrp["Timestamp"].dt.date.value_counts().sort_index().to_string())
assert rsrp["Timestamp"].dt.date.nunique() == 3, "Expected exactly three dates"

[1.4] (DROP) Restrict to complete days 2019-11-02 to 2019-11-04
    before : 2,062,210
    after  : 2,062,146
    removed: 64  (0.003%)

Records per date after restriction:


Timestamp
2019-11-02    698504
2019-11-03    748293
2019-11-04    615349


### B.3 Manufacturer normalisation

Case variants of the same vendor (`htc`/`HTC`, `lenovo`/`Lenovo`/`LENOVO`, `realme`/`Realme`)
plus two corporate aliases — `LGE` is LG's legal entity name, and HMD Global manufactures
Nokia-branded handsets.

Left unnormalised, a single vendor appears as several separate bars.

In [16]:
print(f"Distinct manufacturers before: {rsrp['DeviceManufacturer'].nunique()}")

rsrp["DeviceManufacturer"] = rsrp["DeviceManufacturer"].replace(MANUFACTURER_MAP).str.strip()

print(f"Distinct manufacturers after : {rsrp['DeviceManufacturer'].nunique()}\n")
print("Top 10 by share (%):")
print((rsrp["DeviceManufacturer"].value_counts().head(10) / len(rsrp) * 100).round(2).to_string())

Distinct manufacturers before: 24


Distinct manufacturers after : 20

Top 10 by share (%):
DeviceManufacturer
Samsung     87.04
Huawei       4.93
HTC          1.57
LG           1.32
Xiaomi       1.06
Nokia        0.91
Lenovo       0.73
Motorola     0.68
TCL          0.43
Oppo         0.40


### B.4 Device-name aliasing — canonical names, then collapse  ⭐ NEW IN v3

Records sharing timestamp, coordinates, operator **and** RSRP value, differing only in
`DeviceName`. Three competing explanations:

| Hypothesis | Meaning | Correct response |
|---|---|---|
| **A. Device aliasing** | One handset logged under several model strings | Rename, then collapse |
| **B. Genuine co-location** | Several real users at one spot | Keep all |
| **C. Retransmission** | Same measurement uploaded twice | Remove surplus |

**The test:** within each co-located group, does the RSRP value vary? Identical RSRP with a
differing device name points to A or C; varying RSRP points to B.

---

**Why v3 does more than v2 did.** v2 ran this test, concluded aliasing, and de-duplicated on
`(time, lat, lon, operator, RSRP)` keeping the first row. That removed the duplicate *rows*
— but the survivor kept whichever alias happened to come first, so **both names still existed
across the dataset**:

```
Galaxy J7 Prime   Operator A   12,723 samples   -85.68 dBm
Galaxy On Nxt     Operator A   12,723 samples   -85.68 dBm    <-- same phone, drawn twice
Mi 9T             Operator A    2,462 samples   -74.57 dBm
Redmi K20         Operator A    1,231 samples   -74.57 dBm    <-- same phone, split 2:1
```

Any per-model chart therefore showed one handset as two bars with a fraction of its true
sample count each — and a "minimum samples per model" filter judged it on the halved number.

**v3 fixes the order:** derive a canonical name for each alias group, apply it, *then*
collapse. The survivor now carries the right name and the full count.

In [17]:
grp = rsrp.groupby(COLOCATION_KEY, observed=True).agg(
    n_records=("RSRP", "size"),
    n_distinct_rsrp=("RSRP", "nunique"),
    n_distinct_device=("DeviceName", "nunique"),
)
multi = grp[grp["n_records"] > 1]

identical_rsrp = int((multi["n_distinct_rsrp"] == 1).sum())
varying_rsrp   = int((multi["n_distinct_rsrp"] >  1).sum())

print(f"Co-located groups (size > 1) : {len(multi):,}")
print(f"Records involved             : {int(multi['n_records'].sum()):,}\n")
print(f"Groups with IDENTICAL RSRP : {identical_rsrp:,}  "
      f"({100*identical_rsrp/len(multi):.1f}%)  -> aliasing or retransmission")
print(f"Groups with VARYING RSRP   : {varying_rsrp:,}  "
      f"({100*varying_rsrp/len(multi):.1f}%)  -> genuine co-location")

# Hypothesis B would leave a visible RSRP spread. It does not.
assert varying_rsrp == 0, (
    "Some co-located groups show varying RSRP — genuine co-location is present. "
    "Do NOT collapse blindly; re-run the investigation."
)
print("\nHypothesis A confirmed: every co-located group shares one RSRP value.")

Co-located groups (size > 1) : 66,516
Records involved             : 153,090

Groups with IDENTICAL RSRP : 66,516  (100.0%)  -> aliasing or retransmission
Groups with VARYING RSRP   : 0  (0.0%)  -> genuine co-location

Hypothesis A confirmed: every co-located group shares one RSRP value.


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Build the canonical name map from the data itself.
#
# For every alias group, the canonical name is the variant that appears most often
# across the whole dataset — the one a reader is most likely to recognise. Ties are
# broken alphabetically so the map is deterministic between runs.
# ─────────────────────────────────────────────────────────────────────────────
dup_mask = rsrp.duplicated(subset=COLOCATION_KEY + ["RSRP"], keep=False)
alias_sets = (
    rsrp[dup_mask]
    .groupby(COLOCATION_KEY + ["RSRP"], observed=True)["DeviceName"]
    .agg(lambda s: tuple(sorted(set(s))))
)
alias_sets = alias_sets[alias_sets.map(len) > 1]

global_freq = rsrp["DeviceName"].value_counts()

pair_counts = alias_sets.value_counts()
DEVICE_ALIAS_MAP, alias_rows = {}, []
for names, n_groups in pair_counts.items():
    canonical = sorted(names, key=lambda n: (-int(global_freq.get(n, 0)), n))[0]
    for n in names:
        if n != canonical:
            DEVICE_ALIAS_MAP[n] = canonical
    alias_rows.append({"canonical_name": canonical,
                       "alias_names": " | ".join(n for n in names if n != canonical),
                       "n_names": len(names), "n_groups": int(n_groups)})

alias_table = pd.DataFrame(alias_rows).sort_values("n_groups", ascending=False)

print(f"Alias groups detected : {len(alias_table)}")
print(f"Names to be renamed   : {len(DEVICE_ALIAS_MAP)}\n")
print(alias_table.head(10).to_string(index=False))

Alias groups detected : 28
Names to be renamed   : 38

     canonical_name                                       alias_names  n_names  n_groups
    Galaxy J7 Prime                                     Galaxy On Nxt        2     33404
 Huawei Mate 20 Pro                                       Mate 20 Pro        2     11304
 Huawei P Smart Pro Huawei Y9 Prime 2019 | Huawei Y9S | Y9 Prime 2019        4      5912
           Honor 9A       Huawei Y7 2019 | Huawei Y7S | Y7 Prime 2019        4      3451
Desire 628 Dual Sim                           Htc Desire 628 Dual Sim        2      1871
              Mi 9T                                         Redmi K20        2      1564
              D728W                               Desire 728 Dual Sim        2      1527
     Huawei Y7 2018                              Huawei Y7 Prime 2018        2      1488
        Realme 5Pro                                          Realme Q        2       799
           Honor 7A                                    

In [19]:
# Apply the canonical names BEFORE collapsing, so the surviving row carries the
# right name and the full sample count.
n_renamed = rsrp["DeviceName"].isin(DEVICE_ALIAS_MAP).sum()
rsrp["DeviceName"] = rsrp["DeviceName"].replace(DEVICE_ALIAS_MAP)

log_step(
    "6.2a", f"Rename {len(DEVICE_ALIAS_MAP)} device aliases to a canonical name",
    len(rsrp), len(rsrp),
    f"{n_renamed:,} rows renamed. Without this the same handset survives under two "
    f"names with its samples split between them (Galaxy J7 Prime / Galaxy On Nxt).",
    kind="FIX",
)

rows_before = len(rsrp)
rsrp = rsrp.drop_duplicates(subset=COLOCATION_KEY + ["RSRP"], keep="first").reset_index(drop=True)

log_step(
    "6.2b", "Collapse alias duplicates (dedupe on time + location + operator + RSRP)",
    rows_before, len(rsrp),
    "All co-located groups showed identical RSRP and identical manufacturer, differing "
    "only in DeviceName — regional marketing variants of one handset, each logging the "
    "same physical measurement once per alias.",
)

[6.2a] (FIX) Rename 38 device aliases to a canonical name
    before : 2,062,146
    after  : 2,062,146
    removed: 0  (0.000%)


[6.2b] (DROP) Collapse alias duplicates (dedupe on time + location + operator + RSRP)
    before : 2,062,146
    after  : 1,975,572
    removed: 86,574  (4.198%)


In [20]:
# ── Loud detector ───────────────────────────────────────────────────────────
# The map above was derived from THIS extract. A future extract may introduce a new
# alias pair; if it does, this assertion fails rather than letting a model quietly
# split into two bars again.
residual = rsrp.duplicated(subset=COLOCATION_KEY + ["RSRP"], keep=False)
assert residual.sum() == 0, (
    f"{residual.sum():,} co-located duplicates survived the collapse — "
    "a new alias pattern has appeared. Inspect before trusting per-device numbers."
)

still_split = (
    rsrp.groupby(COLOCATION_KEY, observed=True)["DeviceName"].nunique()
)
assert (still_split > 1).sum() == 0, (
    "Some co-located groups still carry more than one device name — the canonical map "
    "is incomplete. Re-derive it before exporting."
)
print("Detector passed: no residual aliasing, no split models.")

print("\nVerification — the pairs that were split in v2:")
for a, b in [("Galaxy J7 Prime", "Galaxy On Nxt"), ("Mi 9T", "Redmi K20"), ("M9", "One M9")]:
    present = [n for n in (a, b) if n in set(rsrp["DeviceName"])]
    n = int((rsrp["DeviceName"] == present[0]).sum()) if present else 0
    print(f"  {a:<18} / {b:<16} -> surviving name: {present[0] if present else 'none':<18} "
          f"({n:,} samples)")

Detector passed: no residual aliasing, no split models.

Verification — the pairs that were split in v2:


  Galaxy J7 Prime    / Galaxy On Nxt    -> surviving name: Galaxy J7 Prime    (33,485 samples)


  Mi 9T              / Redmi K20        -> surviving name: Mi 9T              (1,564 samples)


  M9                 / One M9           -> surviving name: M9                 (508 samples)


### B.5 Quantised-device detection  ⭐ NEW IN v3

A handset measuring properly reports **40–80 distinct RSRP values** over three days — the
signal changes as its owner moves.

A few models do not. They report the same two or three numbers over and over, which is the
signature of a value read from a lookup table rather than measured off the air. The clearest
case puts **87% of 210,808 readings on exactly three values**.

**These rows are flagged, never deleted.** Two reasons:

1. The reading is wrong, but the handset was genuinely present at that place and time —
   so it still counts as a user for density purposes.
2. The correction changes which operator looks best. That is a finding, not plumbing, and a
   reader must be able to see it both ways.

Every export in Notebook 03 therefore carries a corrected value **and** its uncorrected twin.

In [21]:
pair_stats = (
    rsrp.groupby(["DeviceName", "RadioOperatorName"], observed=True)["RSRP"]
        .agg(n_samples="size",
             n_distinct="nunique",
             top3=lambda s: s.value_counts().head(3).sum())
        .reset_index()
)
pair_stats["top3_share"] = pair_stats["top3"] / pair_stats["n_samples"]
pair_stats["is_quantised"] = (
    (pair_stats["top3_share"] >= QUANT_TOP3_SHARE) &
    (pair_stats["n_samples"]  >= QUANT_MIN_SAMPLES)
)

flagged = pair_stats[pair_stats["is_quantised"]].sort_values("top3_share", ascending=False)

print(f"Model/operator pairs examined (>= {QUANT_MIN_SAMPLES:,} samples): "
      f"{int((pair_stats['n_samples'] >= QUANT_MIN_SAMPLES).sum())}")
print(f"Flagged as quantised: {len(flagged)}\n")

show = flagged[["DeviceName", "RadioOperatorName", "n_samples", "top3_share", "n_distinct"]].copy()
show["top3_share"] = (show["top3_share"] * 100).round(1).astype(str) + "%"
print(show.to_string(index=False))

healthy = pair_stats[(~pair_stats["is_quantised"]) &
                     (pair_stats["n_samples"] >= QUANT_MIN_SAMPLES)]
print(f"\nFor contrast — healthy pairs of comparable size:")
print(f"  median top-3 share   : {healthy['top3_share'].median()*100:.1f}%")
print(f"  median distinct values: {healthy['n_distinct'].median():.0f}")

Model/operator pairs examined (>= 3,000 samples): 88
Flagged as quantised: 6

     DeviceName RadioOperatorName  n_samples top3_share  n_distinct
   Galaxy Note8        Operator A     210808      87.2%          77
             P2        Operator C       4553      68.0%          24
        K5 Note        Operator B       3696      50.5%          43
Galaxy A7(2017)        Operator B       7480      46.4%          66
   Galaxy Note9        Operator A      28561      42.2%          84
 Galaxy S6 Edge        Operator A      18014      41.1%          50

For contrast — healthy pairs of comparable size:
  median top-3 share   : 14.9%
  median distinct values: 63


In [22]:
# Show WHY the worst case is not a coverage reading
worst = flagged.iloc[0]
sample = rsrp[(rsrp["DeviceName"] == worst["DeviceName"]) &
              (rsrp["RadioOperatorName"] == worst["RadioOperatorName"])]["RSRP"]

print(f"{worst['DeviceName']} on {worst['RadioOperatorName']} — value frequency\n")
vc = sample.value_counts().head(6)
for val, n in vc.items():
    print(f"  {val:>5} dBm : {n:>7,}  ({100*n/len(sample):>5.1f}%)  {'#' * int(60*n/len(sample))}")
print(f"\n  ...{sample.nunique()} distinct values in total, but the top 3 hold "
      f"{100*vc.head(3).sum()/len(sample):.1f}% of all readings.")
print("\nNo propagation environment produces that. It is a firmware/SDK reporting fault.")

Galaxy Note8 on Operator A — value frequency

   -107 dBm :  94,439  ( 44.8%)  ##########################
    -71 dBm :  47,253  ( 22.4%)  #############
    -83 dBm :  42,201  ( 20.0%)  ############
   -104 dBm :   5,807  (  2.8%)  #
    -79 dBm :     859  (  0.4%)  
    -77 dBm :     824  (  0.4%)  

  ...77 distinct values in total, but the top 3 hold 87.2% of all readings.

No propagation environment produces that. It is a firmware/SDK reporting fault.


In [23]:
# Attach the flag to every row. No rows are removed.
flag_key = set(zip(flagged["DeviceName"], flagged["RadioOperatorName"]))
rsrp["is_quantised"] = [
    (d, o) in flag_key for d, o in zip(rsrp["DeviceName"], rsrp["RadioOperatorName"])
]

log_step(
    "7.1", f"FLAG {len(flagged)} quantised model/operator pairs (no rows removed)",
    len(rsrp), len(rsrp),
    f"Pairs placing >= {QUANT_TOP3_SHARE:.0%} of readings on their three most common RSRP "
    f"values with >= {QUANT_MIN_SAMPLES:,} samples. Flagged rather than deleted because the "
    f"handset was genuinely present, and because the correction changes the operator ranking "
    f"— a finding that must be demonstrable, not silently applied.",
    kind="FLAG",
)

print(f"\nRows flagged: {int(rsrp['is_quantised'].sum()):,}  "
      f"({100*rsrp['is_quantised'].mean():.1f}% of the clean table)")
print("\nShare of each operator's samples that are flagged:")
print((rsrp.groupby("RadioOperatorName", observed=True)["is_quantised"].mean() * 100)
      .round(1).to_string())

[7.1] (FLAG) FLAG 6 quantised model/operator pairs (no rows removed)
    before : 1,975,572
    after  : 1,975,572
    removed: 0  (0.000%)

Rows flagged: 273,112  (13.8% of the clean table)

Share of each operator's samples that are flagged:
RadioOperatorName
Operator A    23.9
Operator B     2.5
Operator C     1.0


In [24]:
# The impact, stated plainly — this is the headline the flag exists to support.
impact = pd.DataFrame({
    "as_delivered": rsrp.groupby("RadioOperatorName", observed=True)["RSRP"].mean(),
    "corrected":    rsrp[~rsrp["is_quantised"]].groupby("RadioOperatorName", observed=True)["RSRP"].mean(),
}).round(2)
impact["change_dB"] = (impact["corrected"] - impact["as_delivered"]).round(2)
impact["rank_as_delivered"] = impact["as_delivered"].rank(ascending=False).astype(int)
impact["rank_corrected"]    = impact["corrected"].rank(ascending=False).astype(int)

print("Mean RSRP, before and after excluding the flagged handsets\n")
print(impact.to_string())
print("\nOne operator changes rank. That is why this is a toggle and not a silent deletion.")

Mean RSRP, before and after excluding the flagged handsets

                   as_delivered  corrected  change_dB  rank_as_delivered  rank_corrected
RadioOperatorName                                                                       
Operator A               -85.47     -83.56       1.91                  3               2
Operator B               -84.65     -84.79      -0.14                  2               3
Operator C               -80.87     -80.83       0.04                  1               1

One operator changes rank. That is why this is a toggle and not a silent deletion.


---
## Stage C — Derivation & Persistence

Derived fields are computed **last**, after the row set is final. Computing them earlier
wastes effort on rows subsequently removed — and in the case of hour-of-day, would bake in
the pre-timezone-fix error permanently.

In [25]:
rsrp["date"]         = rsrp["Timestamp"].dt.date
rsrp["hour"]         = rsrp["Timestamp"].dt.hour          # 0-23, local time
rsrp["day_of_week"]  = rsrp["Timestamp"].dt.day_name()
rsrp["hour_of_week"] = rsrp["Timestamp"].dt.dayofweek * 24 + rsrp["hour"]

print("Records by hour of day:")
print(rsrp["hour"].value_counts().sort_index().to_string())

Records by hour of day:
hour
0      39212
1      44467
2      31489
3      22826
4      22503
5      30738
6      52336
7      89053
8      95650
9      84961
10     84411
11     95820
12     94302
13     96799
14     89676
15     98859
16    140069
17    133186
18    137899
19    110590
20    104540
21    104128
22    103553
23     68505


### C.2 Spatial bins — degree grid (primary) + H3 (secondary)

**Degree bins are the primary grid.** Three resolutions are pre-computed so a dashboard's
aggregation control can switch between them without recomputation. At Riyadh's latitude
0.01° latitude ≈ 1.11 km and 0.01° longitude ≈ 1.01 km, so a "1 km" cell is a rectangle
about 10% taller than it is wide — immaterial for these charts, and the key is reproducible
anywhere with one line of arithmetic.

**H3 hex keys are carried as a second key.** Hexagons have equal-distance neighbours and no
directional bias, which matters if anyone later does neighbour analysis (spatial smoothing,
contiguity). Carrying both means that switch costs nothing.

In [26]:
for deg, label in [(0.005, "500m"), (0.01, "1km"), (0.02, "2km")]:
    rsrp[f"bin_{label}"] = (
        (rsrp["LocationLatitude"]  / deg).round().astype(int).astype(str) + "_" +
        (rsrp["LocationLongitude"] / deg).round().astype(int).astype(str)
    )
    # Cell centre — what a map should plot, rather than the raw GPS point
    rsrp[f"bin_{label}_lat"] = (rsrp["LocationLatitude"]  / deg).round() * deg
    rsrp[f"bin_{label}_lon"] = (rsrp["LocationLongitude"] / deg).round() * deg
    print(f"{label:>6} bins: {rsrp[f'bin_{label}'].nunique():,} distinct cells")

  500m bins: 4,623 distinct cells


   1km bins: 1,588 distinct cells


   2km bins: 533 distinct cells


In [27]:
if HAS_H3:
    lat = rsrp["LocationLatitude"].to_numpy()
    lon = rsrp["LocationLongitude"].to_numpy()
    for res, label in [(7, "h3_r7"), (8, "h3_r8")]:
        cells = [h3.latlng_to_cell(a, b, res) for a, b in zip(lat, lon)]
        centres = [h3.cell_to_latlng(c) for c in cells]
        rsrp[label]           = cells
        rsrp[f"{label}_lat"]  = np.round([c[0] for c in centres], 6)
        rsrp[f"{label}_lon"]  = np.round([c[1] for c in centres], 6)
        edge_km = h3.average_hexagon_edge_length(res, unit="km")
        print(f"{label}: {rsrp[label].nunique():,} cells   (edge {edge_km:.2f} km)")
else:
    print("h3 not installed — skipping the secondary hex keys.")

h3_r7: 491 cells   (edge 1.41 km)


h3_r8: 2,207 cells   (edge 0.53 km)


In [28]:
# Coverage band — the continuous KPI turned into the five service tiers an
# operations team plans against.
rsrp["coverage_band"] = pd.cut(rsrp["RSRP"], bins=BAND_EDGES,
                               labels=BAND_LABELS, right=False).astype(str)

print("Share of samples per band (%):")
print((rsrp["coverage_band"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())

Share of samples per band (%):
coverage_band
1 Excellent    41.74
2 Good         27.95
3 Fair         17.88
4 Poor         11.47
5 Very Poor     0.95


In [29]:
for col in ["RadioOperatorName", "RadioNetworkGeneration", "RadioConnectionType",
            "DeviceManufacturer", "DeviceName", "day_of_week", "coverage_band"]:
    rsrp[col] = rsrp[col].astype("category")

rsrp["RSRP"] = rsrp["RSRP"].astype("int16")

print(f"Memory after optimisation: {rsrp.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

Memory after optimisation: 556 MB


In [30]:
log_df = show_log()
display(log_df)

final = len(rsrp)
print(f"\nOriginal rows : {ROWS_ORIGINAL:,}")
print(f"Final rows    : {final:,}")
print(f"Removed       : {ROWS_ORIGINAL - final:,}  "
      f"({100*(ROWS_ORIGINAL-final)/ROWS_ORIGINAL:.2f}%)")

# Every removed row must be accounted for in the log.
assert log_df["removed"].sum() == ROWS_ORIGINAL - final, \
    "Logged removals do not reconcile with the total row reduction"
print("\nReconciliation passed — every removed row is accounted for in the log.")

,step,kind,rule,rows_before,rows_after,removed,pct_removed
0,3.2,DROP,Retain 4G + Mobile + data Enabled only,2725353,2250966,474387,17.406
1,6.1,DROP,Drop exact full-row duplicates (all 11 columns...,2250966,2249289,1677,0.075
2,4.1-4.3,DROP,"Retain RSRP within [-140, -40] dBm",2249289,2062210,187079,8.317
3,2.1-2.2,DROP,Retain coordinates within the Riyadh bounding box,2062210,2062210,0,0.000
4,1.1-1.2,DROP,Parse timestamps honouring declared UTC offset...,2062210,2062210,0,0.000
5,1.4,DROP,Restrict to complete days 2019-11-02 to 2019-1...,2062210,2062146,64,0.003
6,6.2a,FIX,Rename 38 device aliases to a canonical name,2062146,2062146,0,0.000
7,6.2b,DROP,Collapse alias duplicates (dedupe on time + lo...,2062146,1975572,86574,4.198
8,7.1,FLAG,FLAG 6 quantised model/operator pairs (no rows...,1975572,1975572,0,0.000



Original rows : 2,725,353
Final rows    : 1,975,572
Removed       : 749,781  (27.51%)

Reconciliation passed — every removed row is accounted for in the log.


In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# The clean table is an INTERMEDIATE — Notebook 03 reads it, Tableau does not.
# Parquet keeps dtypes and categories intact and is ~10x smaller than CSV, but it
# needs pyarrow. If pyarrow is absent we fall back to pickle, which ships with
# Python and preserves dtypes just as faithfully. Notebook 03 accepts either.
# ─────────────────────────────────────────────────────────────────────────────
try:
    INTERMEDIATE = INTERIM_DIR / "rsrp_clean.parquet"
    rsrp.to_parquet(INTERMEDIATE, index=False)
except (ImportError, ValueError) as exc:
    print(f"NOTE: parquet unavailable ({type(exc).__name__}) — writing pickle instead.")
    INTERMEDIATE = INTERIM_DIR / "rsrp_clean.pkl"
    rsrp.to_pickle(INTERMEDIATE)

log_df.to_csv(REF_DIR / "rsrp_cleaning_log.csv", index=False)
tech_mix.to_csv(OUT_DIR / "tech_mix.csv", index=False)
alias_table.to_csv(REF_DIR / "device_alias_map.csv", index=False)

size_mb = INTERMEDIATE.stat().st_size / 1024**2
print(f"interim/   {INTERMEDIATE.name:<22} {len(rsrp):>9,} rows  {size_mb:>5.0f} MB   [for Notebook 03]")
print(f"processed/ tech_mix.csv            {len(tech_mix):>9,} rows                [extract 8 of 9]")
print(f"reference/ rsrp_cleaning_log.csv   {len(log_df):>9,} steps               [audit trail]")
print(f"reference/ device_alias_map.csv    {len(alias_table):>9,} rows                [audit trail]")

interim/   rsrp_clean.parquet     1,975,572 rows     35 MB   [for Notebook 03]
processed/ tech_mix.csv                   15 rows                [extract 8 of 9]
reference/ rsrp_cleaning_log.csv           9 steps               [audit trail]
reference/ device_alias_map.csv           28 rows                [audit trail]


---
## Summary

`interim/rsrp_clean.parquet` (or `.pkl`) — scoped, deduplicated, timezone-corrected, **canonically named**, and
**quantisation-flagged**, with degree bins, H3 keys, and coverage bands.

### Carried forward into Notebook 03

- **`is_quantised` is a flag, not a filter.** Every export must carry a corrected measure and
  its uncorrected twin so the dashboard toggle works.
- **Device names are now canonical.** One row per handset per operator; no split models.
- **Degree bins are primary, H3 secondary.** Both keys travel with every spatial export.
- **`tech_mix.csv` was captured before scoping** — it is the only way to compute the LTE
  attach-rate KPI, since the non-4G rows no longer exist here.

### Next

`02_TrafficVolume_Data_Preparation.ipynb`